# Studi Kasus 1: Predictive Maintenance Mesin Jahit & Estimasi Beban Kerja (Mode Lokal)

Notebook ini merupakan versi rilis publik **100% lokal & teranonimisasi** untuk pilar **Predictive Maintenance** pada lembaga pelatihan vokasi. Seluruh data diambil langsung dari `../data/maintenance_anonymized.csv` tanpa koneksi API eksternal.

**Fokus Analitik (DS / ML / DA / BI):**
1. Memuat dataset servis mesin & fitur *IoT Proxy* (`akumulasi_jp`, `rasio_beban_harian`, `jp_per_servis`) secara lokal.
2. Eksplorasi distribusi kondisi 3-kelas vs 4-kelas serta profil kerusakan per tipe mesin.
3. Replikasi evaluasi *pipeline* `RandomForestClassifier` (3-kelas) dan `RandomForestRegressor` (estimasi JP).
4. Uji inferensi menggunakan *serving engine* lokal (`src/inference.py`).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_maintenance_data
from src.inference import VocationalInferenceEngine

df = load_maintenance_data()
print(f"Dataset lokal termuat: {df.shape[0]} log servis pada {df['asset_id'].nunique()} unit mesin unik")
print(df[['asset_id', 'jenis_mesin', 'usia_mesin_hari', 'akumulasi_jp', 'riwayat_servis_ke', 'kondisi']].head(5).to_string(index=False))

Dataset lokal termuat: 86 log servis pada 46 unit mesin unik
asset_id         jenis_mesin  usia_mesin_hari  akumulasi_jp  riwayat_servis_ke      kondisi
 AST-001 MESIN_JAHIT_LAINNYA              373        737.50                  1         Baik
 AST-001 MESIN_JAHIT_LAINNYA              970        842.24                  2 Rusak Ringan
 AST-002 MESIN_JAHIT_LAINNYA              477          0.00                  1         Baik
 AST-002 MESIN_JAHIT_LAINNYA              878       3041.01                  2 Rusak Ringan
 AST-003 MESIN_JAHIT_LAINNYA              350       4111.84                  1         Baik


## 1. Exploratory Data Analysis (DA / BI Summary)

In [2]:
print("=== Distribusi Target 3-Kelas (0=Baik, 1=Rusak Ringan, 2=Rusak Berat/Afkir) ===")
print(df['kondisi_3_kelas'].value_counts().sort_index().to_string())

print("\n=== Ringkasan per Tipe Mesin ===")
summary_type = (
    df.groupby('jenis_mesin')
      .agg(
          total_log=('asset_id', 'count'),
          unit_unik=('asset_id', 'nunique'),
          rata2_usia_hari=('usia_mesin_hari', 'mean'),
          rata2_jp=('akumulasi_jp', 'mean'),
          persen_berisiko=('butuh_servis', lambda s: round(s.mean() * 100, 1))
      )
      .sort_values('total_log', ascending=False)
      .round(1)
)
print(summary_type.to_string())

=== Distribusi Target 3-Kelas (0=Baik, 1=Rusak Ringan, 2=Rusak Berat/Afkir) ===
kondisi_3_kelas
0    45
1    29
2    12

=== Ringkasan per Tipe Mesin ===
                     total_log  unit_unik  rata2_usia_hari  rata2_jp  persen_berisiko
jenis_mesin                                                                          
HIGH_SPEED                  29         15            805.1    2460.6             51.7
PORTABLE                    27         17            642.7    2198.4             40.7
MESIN_JAHIT_LAINNYA          8          4            618.4    1399.9             37.5
BUTTERFLY_MANUAL             6          4            604.2    1470.2             33.3
SINGER                       6          2            797.2    2745.9             66.7
OBRAS                        4          2            913.5    1690.9             75.0
NECI                         3          1            557.7     527.5             66.7
OVERDECK                     3          1            941.7    2281.7    

## 2. Validasi Pipeline Klasifikasi 3-Kelas & Regresi Beban JP (Lokal)

In [3]:
feature_clf = [
    'usia_mesin_hari', 'akumulasi_jp', 'riwayat_servis_ke',
    'rasio_beban_harian', 'jp_per_servis',
    'tipe_BUTTERFLY_MANUAL', 'tipe_HIGH_SPEED', 'tipe_MESIN_JAHIT_LAINNYA',
    'tipe_NECI', 'tipe_OBRAS', 'tipe_OVERDECK', 'tipe_PORTABLE', 'tipe_SINGER'
]
feature_reg = ['usia_mesin_hari', 'riwayat_servis_ke'] + feature_clf[5:]

X_clf = df[feature_clf]
y_clf = df['kondisi_3_kelas']

X_tr, X_te, y_tr, y_te = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

pipe_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', RandomForestClassifier(
        n_estimators=50, max_depth=3, min_samples_leaf=2,
        class_weight='balanced_subsample', random_state=42
    ))
])
pipe_rf.fit(X_tr, y_tr)
preds = pipe_rf.predict(X_te)
print(f"Hold-out Macro-F1 (Replikasi Lokal): {f1_score(y_te, preds, average='macro'):.4f}")
print(classification_report(y_te, preds, target_names=['Baik', 'Rusak Ringan', 'Rusak Berat/Afkir'], zero_division=0))

# Validasi Regresi Beban JP
X_reg = df[feature_reg]
y_reg = df['akumulasi_jp']
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
pipe_reg = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])
pipe_reg.fit(Xr_tr, yr_tr)
mae_reg = mean_absolute_error(yr_te, pipe_reg.predict(Xr_te))
print(f"Hold-out MAE Estimasi Beban JP: {mae_reg:.2f} JP")

Hold-out Macro-F1 (Replikasi Lokal): 0.5199
                   precision    recall  f1-score   support

             Baik       0.62      0.56      0.59         9
     Rusak Ringan       0.50      0.67      0.57         6
Rusak Berat/Afkir       0.50      0.33      0.40         3

         accuracy                           0.56        18
        macro avg       0.54      0.52      0.52        18
     weighted avg       0.56      0.56      0.55        18

Hold-out MAE Estimasi Beban JP: 1696.60 JP


## 3. Uji Inferensi Artefak Model Ter-deploy (`models/*.pkl`)

In [4]:
engine = VocationalInferenceEngine()
sample_result = engine.predict_maintenance(
    usia_mesin_hari=500,
    akumulasi_jp=2200.0,
    riwayat_servis_ke=1,
    jenis_mesin='HIGH_SPEED'
)
for k, v in sample_result.items():
    print(f"{k}: {v}")

derived_features: {'rasio_beban_harian': 4.3912, 'jp_per_servis': 2200.0}
classification_3class: {'code': 1, 'label': 'Rusak Ringan (Waspada)', 'probabilities': {'Baik': 0.4351, 'Rusak Ringan': 0.4557, 'Rusak Berat / Afkir': 0.1092}}
classification_4class: {'code': 1, 'label': 'Rusak Ringan'}
estimated_akumulasi_jp: 3817.41
recommendation: Jadwalkan inspeksi teknisi & servis preventif sebelum kelas intensif berikutnya.
